In [0]:
dbutils.widgets.text("catalog", "procurement")
catalog = dbutils.widgets.get("catalog")
checkpoint_root = f"/Volumes/{catalog}/ops/checkpoints/silver"


from pyspark.sql import DataFrame, functions as F

TECHNICAL_COLUMNS = ["_source_file", "_batch", "_ingested_at"]


def try_cast(column: str, sql_type: str):
    """Cast that returns NULL for bad values instead of failing (serverless runs in ANSI mode)."""
    return F.expr(f"try_cast({column} AS {sql_type})")


def standardise(column: str):
    """Fix soft defects in codes: ' active ' and 'cc-200' become 'ACTIVE' and 'CC-200'."""
    return F.upper(F.trim(F.col(column)))


def add_reject_reason(df: DataFrame, rules: list) -> DataFrame:
    """reject_reason lists every rule the row failed, separated by '; '. Empty string means valid."""
    failed = [F.when(F.coalesce(condition, F.lit(False)), F.lit(reason)) for reason, condition in rules]
    return df.withColumn("reject_reason", F.concat_ws("; ", *failed))


def write_quarantine(rejected: DataFrame, entity: str, key_column: str) -> None:
    """Insert rejected rows into ops.quarantine. Insert-only MERGE on a hash, so a replay adds nothing."""
    rows = (
        rejected.select(
            F.lit(entity).alias("entity"),
            F.col(key_column).alias("business_key"),
            "reject_reason",
            F.col("_raw").alias("raw_record"),
            "_source_file",
            "_batch",
            F.current_timestamp().alias("quarantined_at"),
        )
        .withColumn("quarantine_id", F.sha2(F.concat_ws("|", "entity", "raw_record", "_source_file"), 256))
        .dropDuplicates(["quarantine_id"])
    )
    rows.createOrReplaceTempView("quarantine_updates")
    rejected.sparkSession.sql(f"""
        MERGE INTO {catalog}.ops.quarantine AS t
        USING quarantine_updates AS s
        ON t.quarantine_id = s.quarantine_id
        WHEN NOT MATCHED THEN INSERT *
    """)


def split_valid(cleaned: DataFrame, rules: list, entity: str, key_column: str, valid_view: str) -> None:
    """Validate, quarantine the failures, and register the valid rows as a temp view."""
    checked = add_reject_reason(cleaned, rules)
    write_quarantine(checked.filter("reject_reason != ''"), entity, key_column)
    checked.filter("reject_reason = ''").drop("reject_reason", "_raw").createOrReplaceTempView(valid_view)


def run_incremental(source_table: str, checkpoint_name: str, process_batch) -> None:
    """Stream new rows from a bronze table through process_batch, then stop."""
    query = (
        spark.readStream.table(source_table)
        .writeStream
        .foreachBatch(process_batch)
        .option("checkpointLocation", f"{checkpoint_root}/{checkpoint_name}")
        .trigger(availableNow=True)
        .start()
    )
    query.awaitTermination()
